# COGS 188 - Project Proposal

## Learning to Drive: Autonomous Navigation in MetaDrive


# Project Description

We will train a driving agent from scratch in MetaDrive using reinforcement learning, then demonstrate the trained model locally. Our main experiment tests whether varied training scenarios improve driving on unseen roads and obstacle layouts.


# Names

- Zahir Ali
- Yasir Rizvi
- Muhammad Omer


# Abstract

We will train an AI agent from scratch to drive a simulated car in MetaDrive. Using reinforcement learning, the agent will learn to follow a route, stay on the road, and avoid obstacles. Training will run on UCSD-provided computing resources, ideally an H100 GPU if available, and the trained model will be demonstrated and recorded on an M2 Pro Mac. To extend the project beyond a basic driving demonstration, we will test whether training on varied roads and obstacle configurations improves performance on unseen scenarios. We will compare two training setups using the same algorithm and training budget, then evaluate route completion, collisions, and road departures.


# Background

Autonomous driving combines sequential decision-making with continuous vehicle control. MetaDrive provides configurable driving scenarios and vehicle dynamics for studying how learned policies generalize across environments. <sup><a href="#ref1">[1]</a></sup>

We will use MetaDrive's built-in observations of vehicle state, navigation, and LiDAR-like surroundings. These inputs let us study control and obstacle avoidance without adding a separate camera-perception task. <sup><a href="#ref2">[2]</a></sup>

Proximal Policy Optimization (PPO) learns a policy by repeatedly collecting experience and updating a clipped policy objective. It supports the continuous controls needed for this project. <sup><a href="#ref3">[3]</a></sup>

Our hypothesis is that a policy exposed to more varied roads and obstacle configurations will generalize better than one repeatedly trained on a small fixed set. This experiment connects reinforcement learning, neural-network function approximation, and validation-based model selection. We will confirm the specific connections to covered lectures with the instructor.


# Problem Statement

We will learn a policy $a_t=\pi_\theta(o_t)$ that maps vehicle, navigation, and obstacle observations to continuous steering and acceleration/braking. The car must reach its destination without collision or road departure and within a fixed time limit.

The environment will contain multi-segment roads, curves, lane boundaries, static obstacles, and light background traffic. Routes will have a feasible path to the destination. Continuous controls interact with vehicle dynamics over long sequences, making this more substantial than a discrete grid or a single simplified road.

**Research question:** Does training on varied road layouts and obstacle configurations improve safe completion on unseen scenarios compared with training on a small fixed set?

The problem is measurable through simulator events and route progress, and reproducible through saved scenario configurations, random seeds, software versions, and model checkpoints.


# Data

**Source:** Experience generated directly in [MetaDrive](https://github.com/metadriverse/metadrive); no external dataset is required.

- **Size:** Target approximately one million transitions per final training run, adjusted after profiling; training will exceed 10,000 transitions. The observation vector will contain more than 10 features, with exact dimensions recorded after setup.
- **One observation:** Vehicle motion, navigation information, and LiDAR-like measurements at one simulation step. Each transition records the observation, action, reward, next observation, and termination status.
- **Critical variables:** Speed, heading, road-boundary distances, route checkpoints, obstacle measurements, steering, and acceleration/braking.
- **Handling:** Use MetaDrive's observation representation, check finite values and action bounds, and save scenario identifiers and training settings. Consecutive transitions are correlated, so sample count alone will not establish generalization.
- **Splits:** Separate training, validation, and test scenario lists. Test scenarios will never guide hyperparameter selection or checkpoint selection.

We will use simulator-provided sensors rather than camera images; rendered simulation will remain available for viewing and recording.


# Proposed Solution

We will train a small neural-network policy with **PPO**, starting all weights randomly. We may use an existing PPO implementation, while building the environment configuration, training workflow, and experiments ourselves. The reward will encourage route progress and penalize collisions and road departures; its settings will be recorded. <sup><a href="#ref4">[4]</a></sup>

**Model selection:** Compare learning rates of **0.0001, 0.0003, and 0.001**, each with the same short training budget. Select using safe completion on separate validation scenarios, with collision rate as a tie-breaker. Freeze the chosen settings before the final comparison.

<table align="left" style="margin-left: 0; margin-right: auto; text-align: left;">
<thead><tr>
<th align="left" style="text-align: left;">Setup</th>
<th align="left" style="text-align: left;">Training experience</th>
</tr></thead>
<tbody>
<tr>
<td align="left" style="text-align: left;">Baseline</td>
<td align="left" style="text-align: left;">A small fixed set of road and obstacle scenarios</td>
</tr>
<tr>
<td align="left" style="text-align: left;">Varied training</td>
<td align="left" style="text-align: left;">A larger set of different road layouts and obstacle configurations</td>
</tr>
</tbody>
</table>
<div style="clear: both;"></div>


Both setups will use the same network, observations, reward, PPO settings, background-traffic settings, and environment-step budget. Scenario diversity is the intended experimental difference.

Training will use UCSD DSMLP, ideally an H100 if allocated; other available GPUs are acceptable. We will checkpoint regularly and profile CPU simulation throughput. Cluster execution must be verified before long training runs. <sup><a href="#ref5">[5]</a></sup>

We will download the trained models to an **M2 Pro Mac with 16 GB RAM** for local MetaDrive demonstrations and video recording. Local compatibility will be checked during the first week. State-based training does not require rendering camera observations.


# Evaluation Metrics

We plan to test both setups on the same **100 unseen scenarios**, with **three independent training runs** per setup if resources permit. Targets will be confirmed after the pilot.

For $E$ attempted episodes:

<table align="left" style="margin-left: 0; margin-right: auto; text-align: left;">
<thead><tr>
<th align="left" style="text-align: left;">Metric</th>
<th align="left" style="text-align: left;">Definition</th>
</tr></thead>
<tbody>
<tr>
<td align="left" style="text-align: left;">Safe completion rate</td>
<td align="left" style="text-align: left;">$100\times S/E$, where $S$ counts arrivals before timeout without collision or road departure</td>
</tr>
<tr>
<td align="left" style="text-align: left;">Collision rate</td>
<td align="left" style="text-align: left;">$100\times C/E$, where $C$ counts episodes with a collision</td>
</tr>
<tr>
<td align="left" style="text-align: left;">Road-departure rate</td>
<td align="left" style="text-align: left;">$100\times D/E$, where $D$ counts episodes leaving the permitted road</td>
</tr>
<tr>
<td align="left" style="text-align: left;">Route progress</td>
<td align="left" style="text-align: left;">$\frac{1}{E}\sum_e p_e$, where $p_e$ is the fraction of the route completed</td>
</tr>
</tbody>
</table>
<div style="clear: both;"></div>


Episodes end on arrival, collision, road departure, or timeout. Stopping indefinitely will not count as success. We will report results across training runs, learning curves, and representative failure cases. The main comparison is safe completion on unseen scenarios; final test outcomes will not be used for tuning.


# Ethics & Privacy


This project demonstrates driving only within MetaDrive. Simulated success does not establish real-world safety or camera-based driving capability, and we will not deploy the policy on a real vehicle.

Limited scenario coverage may produce brittle policies. We will document failures, avoid presenting selected videos as proof of general reliability, and report aggregate evaluation results. Simulator-generated data avoids collecting personal driving imagery. All reused software will be credited and used according to its license.


# Team Expectations


- **Zahir Ali:** MetaDrive setup, scenario configuration, and recording.
- **Yasir Rizvi:** PPO training and UCSD cluster execution.
- **Muhammad Omer:** Evaluation, plots, and experiment tracking.
- These are proposed responsibilities for team confirmation. All members will contribute to the report and review results.
- Meet twice weekly; confirm meeting times and a shared communication channel before training begins.
- Track tasks and experiment settings in a shared repository. Flag blocked work within 24 hours so responsibilities can be adjusted.
- Resolve disagreements through a small pilot where practical, otherwise by majority decision. Freeze evaluation rules before final testing.


# Project Timeline Proposal


The plan assumes approximately four weeks remaining. Exact dates will follow the official course deadline.

<table align="left" style="margin-left: 0; margin-right: auto; text-align: left;">
<thead><tr>
<th align="left" style="text-align: left;">Meeting / period</th>
<th align="left" style="text-align: left;">Meeting time</th>
<th align="left" style="text-align: left;">Completed before meeting</th>
<th align="left" style="text-align: left;">Discuss at meeting</th>
</tr></thead>
<tbody>
<tr>
<td align="left" style="text-align: left;">Week 1</td>
<td align="left" style="text-align: left;">TBD</td>
<td align="left" style="text-align: left;">Verify local and cluster execution; configure scenarios; profile training</td>
<td align="left" style="text-align: left;">Confirm course connections, roles, splits, and training budget</td>
</tr>
<tr>
<td align="left" style="text-align: left;">Week 2</td>
<td align="left" style="text-align: left;">TBD</td>
<td align="left" style="text-align: left;">Compare three learning rates on validation scenarios; begin both final setups</td>
<td align="left" style="text-align: left;">Select and freeze PPO settings; review learning curves</td>
</tr>
<tr>
<td align="left" style="text-align: left;">Week 3</td>
<td align="left" style="text-align: left;">TBD</td>
<td align="left" style="text-align: left;">Finish repeated training runs; evaluate locked test scenarios</td>
<td align="left" style="text-align: left;">Compare metrics and analyze failures</td>
</tr>
<tr>
<td align="left" style="text-align: left;">Week 4</td>
<td align="left" style="text-align: left;">TBD</td>
<td align="left" style="text-align: left;">Draft report, results plots, reproducibility instructions, and recorded demo</td>
<td align="left" style="text-align: left;">Review limitations; finalize and submit</td>
</tr>
</tbody>
</table>
<div style="clear: both;"></div>


**Final deliverables:** Training/evaluation code, saved environment configurations, model checkpoints, a report explaining model selection and results, and a recorded comparison of the driving policies.


# Footnotes

<a id="ref1"></a>1. Li, Q., et al. (2022). *MetaDrive: Composing Diverse Driving Scenarios for Generalizable Reinforcement Learning.* [Paper](https://arxiv.org/abs/2109.12674).

<a id="ref2"></a>2. MetaDrive contributors. *Observations.* [Official documentation](https://metadrive-simulator.readthedocs.io/en/latest/obs.html).

<a id="ref3"></a>3. Schulman, J., et al. (2017). *Proximal Policy Optimization Algorithms.* [Paper](https://arxiv.org/abs/1707.06347).

<a id="ref4"></a>4. MetaDrive contributors. *Reward, Cost, Termination, and Step Information.* [Official documentation](https://metadrive-simulator.readthedocs.io/en/latest/reward_cost_done.html).

<a id="ref5"></a>5. UC San Diego. *Data Science & Machine Learning Platform.* [Access information](https://edtech.ucsd.edu/instructional-tools/dsmlp-datahub/index.html); [resource status](https://datahub.ucsd.edu/hub/status).
